In [1]:
import pandas as pd
import os
import shutil
os.getcwd()

'/mnt/hum01-home01/p88346bn/test/project/ctgan-bbb/code'

In [19]:
data1 = pd.read_csv('evals_best_pbb.csv',header=None)
data1.columns = ['dataset','loss','method','kld_weight','mc_sample_train','bayes','sim','mc_sample',
                 'roc_uni','roc_bi','cio','utility','risk']
# data1.loc[(data1['method']=='CTGAN') & (data1['kld_weight'] != '1'),'method'] = 'bbb'
data1['risk'] = data1['risk'].clip(0, None)
data1['score'] = (data1['utility'] + (1-data1['risk']))/2
data1.head()

,dataset,loss,method,kld_weight,mc_sample_train,bayes,sim,mc_sample,roc_uni,roc_bi,cio,utility,risk,score
0,UK,wasserstein,CTGAN,1,1,False,0,1,0.697993,0.550969,0.153020,0.467327,0.329722,0.568802
1,UK,wasserstein,CTGAN,1,1,False,0,2,0.687062,0.538334,0.187491,0.470962,0.339352,0.565805
2,UK,wasserstein,CTGAN,1,1,False,0,5,0.677014,0.523570,0.139517,0.446700,0.357489,0.544605
3,UK,wasserstein,CTGAN,1,1,False,0,10,0.674035,0.516755,0.136479,0.442423,0.365465,0.538479
4,UK,wasserstein,CTGAN,1,1,False,1,1,0.694570,0.547560,0.191436,0.477856,0.326527,0.575664


In [20]:
data_max = data1.groupby(['dataset','loss','method','kld_weight','mc_sample_train','bayes','mc_sample'])['score'].max().reset_index()
data_max = data_max.merge(data1[['sim','roc_uni','roc_bi','cio','utility','risk','score']],on='score')
data_max.head()

,dataset,loss,method,kld_weight,mc_sample_train,bayes,mc_sample,score,sim,roc_uni,roc_bi,cio,utility,risk
0,Adult,vanilla,CTGAN,1,1,False,1,0.657798,3,0.726061,0.551408,0.232762,0.503410,0.187814
1,Adult,vanilla,CTGAN,1,1,False,2,0.605222,3,0.726454,0.553753,0.273001,0.517736,0.307293
2,Adult,vanilla,CTGAN,1,1,False,5,0.578240,0,0.701904,0.533301,0.280875,0.505360,0.348880
3,Adult,vanilla,CTGAN,1,1,False,10,0.547083,1,0.688078,0.519724,0.191906,0.466570,0.372404
4,Adult,vanilla,bbb,1m,2,True,1,0.742069,0,0.768046,0.532842,0.151524,0.484137,0.000000


In [21]:
bbb_f = []
for i in range(len(data_max)):
    if data_max['method'][i] == 'CTGAN':
        bbb_f.append(data_max['loss'][i])
    elif data_max['method'][i] == 'bbb':
        bbb_f.append('bbb'+'-'+data_max['loss'][i][0])
    else:
        bbb_f.append(data_max['method'][i])
data_max['bbb_function'] = bbb_f

In [51]:
data_max.columns, data2.columns

(Index(['dataset', 'loss', 'method', 'kld_weight', 'mc_sample_train', 'bayes',
        'mc_sample', 'score', 'sim', 'roc_uni', 'roc_bi', 'cio', 'utility',
        'risk', 'bbb_function'],
       dtype='object'),
 Index(['dataset', 'discbayes', 'bbb_function', 'final_val', 'Utility.x',
        'Risk.x', 'mc_sample_train', 'mc_sample', 'kl_weight', 'Utility.y',
        'Risk.y', 'id'],
       dtype='object'))

In [22]:
data2 = pd.read_csv('best_bbb_pbb_seed_1.csv')
final_data_sim = data_max.merge(data2[['dataset','bbb_function','mc_sample']], on=['dataset','bbb_function','mc_sample'])

In [44]:
final_data_sim

,dataset,loss,method,kld_weight,mc_sample_train,bayes,mc_sample,score,sim,roc_uni,roc_bi,cio,utility,risk,bbb_function
0,Adult,vanilla,CTGAN,1,1,False,1,0.657798,3,0.726061,0.551408,0.232762,0.503410,0.187814,vanilla
1,Adult,vanilla,bbb,1m,2,True,1,0.742069,0,0.768046,0.532842,0.151524,0.484137,0.000000,bbb-v
2,Adult,vanilla,fclassic,1m,5,True,1,0.741083,0,0.759986,0.526269,0.178352,0.488202,0.006037,fclassic
3,Adult,vanilla,flamb,2mi,2,True,2,0.741021,0,0.761053,0.523172,0.161900,0.482041,0.000000,flamb
4,Adult,vanilla,fquad,2mi,5,True,1,0.736887,2,0.753417,0.522339,0.145569,0.473775,0.000000,fquad
5,Adult,wasserstein,CTGAN,1,1,False,1,0.613082,4,0.726867,0.560372,0.263199,0.516813,0.290648,wasserstein
6,Adult,wasserstein,bbb,2mi,1,True,1,0.756200,0,0.798434,0.586284,0.166772,0.517163,0.004763,bbb-w
7,Canada,vanilla,CTGAN,1,1,False,1,0.664376,2,0.751137,0.582401,0.259900,0.531146,0.202393,vanilla
8,Canada,vanilla,bbb,2mi,1,True,1,0.725016,4,0.794189,0.604588,0.081845,0.493541,0.043508,bbb-v
9,Canada,vanilla,fclassic,2mi,5,True,1,0.716883,0,0.833120,0.632095,0.093936,0.519717,0.085951,fclassic


In [40]:
final_data_sim.to_csv('final_best_bbb.csv',index=False)

In [24]:
import shutil
for i in range(len(final_data_sim)):
    filename1 = f"best_{final_data_sim['dataset'][i]}_{final_data_sim['method'][i]}_{final_data_sim['loss'][i]}_{final_data_sim['mc_sample'][i]}_{final_data_sim['sim'][i]}.csv"
    src = f'{os.getcwd()}/data_sim_20240528/{filename1}'
    dst = f'{os.getcwd()}/best_for_vis/{filename1}'
#     filename1, src, dst
    shutil.copyfile(src, dst)

In [38]:
data_mcmc = pd.read_csv(os.getcwd()+'/mcmc_synth_data_res.csv')
data_mcmc.columns = ['dataset','method','sample','sim',
                 'roc_uni','roc_bi','cio','utility','risk']
data_mcmc['risk'] = data_mcmc['risk'].clip(0, None)
data_mcmc['score'] = (data_mcmc['utility'] + (1-data_mcmc['risk']))/2

In [46]:
data_max_mcmc = data_mcmc.groupby(['dataset','method','sample'])['score'].max().reset_index()
data_max_mcmc = data_max_mcmc.merge(data_mcmc[['sim','roc_uni','roc_bi','cio','utility','risk','score']],on='score')
data_max_mcmc

,dataset,method,sample,score,sim,roc_uni,roc_bi,cio,utility,risk
0,Adult,ASGHMC,bma,0.661960,3,0.549462,0.353223,0.069075,0.323920,0.000000
1,Adult,ASGHMC,concat,0.661376,4,0.541751,0.347280,0.079227,0.322753,0.000000
2,Adult,ASGHMC,wa,0.662556,0,0.540174,0.349200,0.085961,0.325112,0.000000
3,Adult,PSGLD,bma,0.721432,4,0.757813,0.516847,0.104087,0.459582,0.016719
4,Adult,PSGLD,concat,0.727350,2,0.745107,0.511188,0.107807,0.454701,0.000000
5,Adult,PSGLD,wa,0.728370,3,0.747878,0.514933,0.107409,0.456740,0.000000
6,Canada,ASGHMC,bma,0.564265,1,0.520736,0.360007,0.105809,0.328851,0.200320
7,Canada,ASGHMC,concat,0.728258,0,0.775991,0.577912,0.015647,0.456517,0.000000
8,Canada,ASGHMC,wa,0.731121,0,0.776021,0.577667,0.033036,0.462241,0.000000
9,Canada,PSGLD,bma,0.719123,0,0.760498,0.553826,0.000413,0.438246,0.000000


In [47]:
data_max_mcmc.to_csv('final_best_mcmc.csv',index=False)

In [50]:
import shutil
for i in range(len(data_max_mcmc)):
    filename1 = f"best_{data_max_mcmc['dataset'][i]}_{data_max_mcmc['method'][i]}_vanilla_{data_max_mcmc['sample'][i]}_{data_max_mcmc['sim'][i]}.csv"
    src = f'{os.getcwd()}/data_sim_20240528/{filename1}'
    dst = f'{os.getcwd()}/best_for_vis/{filename1}'
#     print(filename1, src, dst)
    shutil.copyfile(src, dst)